# E4: seed 3

All three reviewers said three seeds is not enough, finding U4. With three runs
there are two degrees of freedom, so the 95% upper bound on the true standard
deviation is roughly six times the sample one: the paper's "48 times the
training noise" could honestly be eight times. This takes the sweep to five.

The corpus is built once from `kaggle.yaml` and every seed trains on that same
`splits.parquet`, so what this measures is training variance and not split
variance. That distinction matters, because `splits.py` seeds its own RNG from
`project.seed`, and rebuilding the corpus per seed would have quietly turned a
seed sweep into a sweep over partitions as well.

## Settings

| Setting | Value |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` |
| **Persistence** | `Files only` |

**Save Version -> Save & Run All (Commit).** Budget about 10 h, which is also
the `--max-hours` the trainer is given, so it stops after the last epoch that
fits rather than being killed mid-epoch. Re-run this notebook with the previous
version's output attached to continue where it stopped.

One seed per notebook, deliberately. A single notebook training both seeds runs
them back to back and takes twice the wall-clock for no benefit, whereas two
notebooks on two accounts finish together.


In [ ]:
import os, sys, time, subprocess, shutil, pathlib, json

T0 = time.time()
def elapsed(label=""):
    m = (time.time() - T0) / 60
    print(f"[{m:6.1f} min] {label}", flush=True)

def run(cmd):
    """Run a stage and stop the notebook if it fails, rather than letting the
    next cell train on whatever stale data is lying around."""
    print(">>", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, "-u", *[str(c) for c in cmd]])
    if r.returncode != 0:
        raise SystemExit(f"FAILED: {' '.join(str(c) for c in cmd)}")

print("Python", sys.version.split()[0])
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 in the right panel.")

## 1. Code

In [ ]:
WORK = pathlib.Path("/kaggle/working/project")
WORK.mkdir(parents=True, exist_ok=True)

def find_aicd():
    root = pathlib.Path("/kaggle/input")
    if not root.exists():
        return None
    for cand in root.rglob("aicd"):
        if (cand / "config.py").exists() and (cand / "models").is_dir():
            return cand
    return None

src = find_aicd()
if src is None:
    raise SystemExit(
        "aicd/ not found under /kaggle/input.\n"
        "Right panel -> Input -> Add Input -> Datasets, then add the dataset\n"
        "you created from aicd-code.zip.")

dest = WORK / "aicd"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(src, dest)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code ->", dest)
if not (dest / "data" / "exposure_arms.py").exists():
    raise SystemExit("This code dataset predates E1. Re-upload aicd-code.zip.")

## 2. Dependencies

In [ ]:
pkgs = ["xgboost", "tree-sitter", "tree-sitter-language-pack",
        "datasets", "shap", "pyyaml", "scikit-learn", "pyarrow", "datasketch"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import importlib
for m in ["xgboost", "sklearn", "transformers", "datasets", "yaml", "datasketch"]:
    mod = importlib.import_module(m)
    print(f"  ok  {m:14s} {getattr(mod, '__version__', '')}")
elapsed("deps")

## 3. Resume, if a previous run is attached

In [ ]:
# Resume support. A fresh Kaggle session starts with an empty working
# directory, so `--resume` alone finds nothing, prints "starting fresh" and
# silently retrains from epoch 0. That is how seven hours disappear without
# anyone noticing, so the checkpoint is restored explicitly here.
#
# The corpus is restored too, and that matters more than it looks. Rebuilding
# is deterministic given the seed but NOT across library versions: the same
# pipeline produced 417,645 rows earlier and 417,431 today, a parse-filter
# difference. Resuming a checkpoint onto a corpus it was not trained on would
# be quietly wrong, so if a previous split is available we use it and skip the
# rebuild entirely.
#
# Attach the previous version's output: Input -> Add Input -> Notebook Output.
import torch
art = WORK / "aicd" / "artifacts"
(art / "data").mkdir(parents=True, exist_ok=True)
CKPT = "branch_a_seed3_ckpt.pt"

def newest(pattern):
    hits = sorted(pathlib.Path("/kaggle/input").rglob(pattern),
                  key=lambda q: q.stat().st_mtime, reverse=True)
    return hits[0] if hits else None

sp = newest("splits.parquet")
RESTORED_SPLITS = False
if sp is not None:
    shutil.copy(sp, art / "data" / "splits.parquet")
    import pandas as _pd
    _n = len(_pd.read_parquet(art / "data" / "splits.parquet", columns=["label"]))
    print(f"restored splits.parquet from {sp}  ({_n:,} rows)")
    RESTORED_SPLITS = True
else:
    print("no previous splits.parquet found; the corpus will be built fresh")

ck = newest(CKPT)
if ck is None:
    print(f"no {CKPT} under /kaggle/input -- this will train from epoch 0.")
    print("If you meant to resume, attach the previous version's output.")
else:
    shutil.copy(ck, art / CKPT)
    _c = torch.load(art / CKPT, map_location="cpu", weights_only=False)
    print(f"restored {CKPT} from {ck}")
    print(f"  holds epoch {_c['epoch']}, so training resumes at epoch {_c['epoch'] + 1}")
    if not RESTORED_SPLITS:
        raise SystemExit(
            "A checkpoint was restored but its corpus was not. Rebuilding may "
            "produce a different split than the one this checkpoint was "
            "trained on, which would make the resumed run unsound. Attach the "
            "previous output so splits.parquet comes with it.")


## 3. Build the corpus

One train shard, exactly the original GPU build: 493,850 raw rows filtering to 417,645, of which 196,854 are training.

In [ ]:
CFG = "kaggle.yaml"

# One train shard, not three. One shard plus dev and test is 493,850 raw rows
# which filter to the 417,645 of the original GPU build, with 196,854 of them
# training. Three shards is the matched-scale corpus and gives roughly 545,000
# training rows, which is a different experiment.
if RESTORED_SPLITS:
    print("corpus restored from the previous run; skipping the rebuild so the")
    print("resumed model continues on exactly the data it was trained on.")
    import pandas as pd
    _sp = pd.read_parquet(WORK / "aicd" / "artifacts" / "data" / "splits.parquet",
                          columns=["split"])
    rows = int((_sp["split"] == "train").sum())
    print(f"training rows: {rows:,}")
else:
    run(["-m", "aicd.data.download", "--config", CFG, "--train-shards", "1"])
    elapsed("downloaded 1 shard")

    for stage in ["normalize", "filter", "splits"]:
        run(["-m", f"aicd.data.{stage}", "--config", CFG])
        elapsed(stage)

    run(["-m", "pytest", "aicd/tests/", "-q"])
    elapsed("integrity tests passed")

    sp_report = json.load(open(WORK / "aicd" / "eval" / "reports" / "splits.json"))
    rows = sp_report["train"]["rows"]
    print(f"\ntraining rows: {rows:,}   (expected 196,854)")
    if abs(rows - 196854) > 5000:
        raise SystemExit(
            f"Got {rows:,} training rows, expected about 196,854. This notebook "
            "must reproduce the original GPU build exactly, or the arms are not "
            "comparable with the existing model.")

## 4. Train seed 3

In [ ]:
run(["-m", "aicd.models.modernbert_triplet",
     "--config", "kaggle_seed3.yaml",
     "--tag", "seed3", "--max-hours", "10", "--resume"])
elapsed("seed 3 done")

## 5. Spread across every seed present

In [ ]:
import numpy as np
rep = WORK / "aicd" / "eval" / "reports"

runs = {"seed 20260818 (paper)": {"s1_in_distribution": 0.8977,
                                  "s5_compound": 0.2378}}
for seed in [1, 2, 3, 4]:
    f = rep / f"branch_a_seed{seed}.json"
    if f.exists():
        r = json.load(open(f))["slices"]
        runs[f"seed {seed}"] = {k: r[k]["macro_f1"]
                                for k in ["s1_in_distribution", "s5_compound"]
                                if k in r}

print(f"{'run':26s} {'S1':>9s} {'S5':>9s}")
print("-" * 47)
for name, v in runs.items():
    print(f"{name:26s} {v.get('s1_in_distribution', float('nan')):9.4f} "
          f"{v.get('s5_compound', float('nan')):9.4f}")

s1 = [v["s1_in_distribution"] for v in runs.values() if "s1_in_distribution" in v]
s5 = [v["s5_compound"] for v in runs.values() if "s5_compound" in v]
if len(s1) > 2:
    drops = [a - b for a, b in zip(s1, s5)]
    print()
    print(f"n = {len(s1)} seeds")
    print(f"S1    mean {np.mean(s1):.4f}  sd {np.std(s1, ddof=1):.4f}")
    print(f"S5    mean {np.mean(s5):.4f}  sd {np.std(s5, ddof=1):.4f}")
    print(f"drop  mean {np.mean(drops):.4f}  sd {np.std(drops, ddof=1):.4f}")
    print()
    print("Report the PAIRED drop with its interval, not the ratio of a drop")
    print("to a standard deviation. That is reviewer finding 4.3: with three")
    print("seeds the 48x claim was not supportable, and with five the honest")
    print("statement is a mean and an interval that excludes zero.")

## 6. Save

In [ ]:
OUT = pathlib.Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

reports = WORK / "aicd" / "eval" / "reports"
if reports.exists():
    shutil.copytree(reports, OUT / "reports", dirs_exist_ok=True)

art = WORK / "aicd" / "artifacts"
npy = OUT / "arrays"
npy.mkdir(exist_ok=True)
n = 0
for f in art.glob("proba_a*.npy"):
    shutil.copy(f, npy / f.name); n += 1
for name in ("arm_report.json", "splits_arms.parquet"):
    p = art / "data" / name
    if p.exists() and p.stat().st_size < 200e6:
        shutil.copy(p, OUT / name)
masks = art / "data" / "arm_masks"
if masks.exists():
    shutil.copytree(masks, OUT / "arm_masks", dirs_exist_ok=True)
if (art / "kaggle").exists():
    for f in (art / "kaggle").glob("*"):
        if f.is_file() and f.stat().st_size < 200e6:
            shutil.copy(f, npy / f.name); n += 1

shutil.make_archive("/kaggle/working/results", "zip", OUT)
print(f"copied {n} arrays")
print("-> /kaggle/working/results.zip  (Output tab)")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.stat().st_size/1024:8.0f} KB  {p.relative_to(OUT)}")
elapsed("saved")